#### Task 1: Build a Fault-Tolerant Multi-Source ETL Pipeline with Conflict Resolution

This notebook pulls data from JSONPlaceholder and a local CSV file, handles API errors, merges the sources, cleans the data, resolves conflicts, and saves the final output to SQLite and CSV.

In [1]:
import pandas as pd
import requests
import mysql.connector

from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from concurrent.futures import ThreadPoolExecutor, as_completed

In [2]:
BASE_URL = "https://jsonplaceholder.typicode.com/"

In [3]:
# Create session
session = requests.Session()

# Retry configuration
retry_strategy = Retry(
    total=3,
    backoff_factor=1,
    status_forcelist=[500, 502, 503, 504]
)

# Attach retry logic to session
adapter = HTTPAdapter(max_retries=retry_strategy)

session.mount("http://", adapter)
session.mount("https://", adapter)

In [4]:
# Fetch data with error handling
def fetch_api_data(endpoint):
    try:
        url = f"{BASE_URL}{endpoint}"
        response = session.get(url, timeout=5)
        response.raise_for_status()

        print(f"Fetched successfully: {endpoint}")
        return endpoint, response.json()

    except requests.exceptions.Timeout:
        print(f"Timeout Error: {endpoint}")
    except requests.exceptions.HTTPError as e:
        print(f"HTTP Error {endpoint}: {e}")
    except requests.exceptions.RequestException as e:
        print(f"Request Error {endpoint}: {e}")

    return endpoint, None


# Parallel API Fetching
endpoints = ["/users", "/posts"]

results = {}

with ThreadPoolExecutor(max_workers=2) as executor:
    futures = [executor.submit(fetch_api_data, ep) for ep in endpoints]

    for future in as_completed(futures):
        endpoint, data = future.result()
        results[endpoint] = data

Fetched successfully: /users
Fetched successfully: /posts


In [5]:
# Use results from the parallel fetch step
users_data = results.get("/users")
posts_data = results.get("/posts")

In [6]:
# User DataFrame Creation
if users_data:

    # Normalize nested JSON
    df_users = pd.json_normalize(users_data)

    # Select required columns
    df_users = df_users[
        ['id', 'name', 'email', 'address.city']
    ]

    # Rename columns
    df_users.columns = [
        'user_id',
        'name',
        'email',
        'city'
    ]

    print("Users DataFrame:")
    display(df_users.head())

Users DataFrame:


,user_id,name,email,city
0,1,Leanne Graham,Sincere@april.biz,Gwenborough
1,2,Ervin Howell,Shanna@melissa.tv,Wisokyburgh
2,3,Clementine Bauch,Nathan@yesenia.net,McKenziehaven
3,4,Patricia Lebsack,Julianne.OConner@kory.org,South Elvis
4,5,Chelsey Dietrich,Lucio_Hettinger@annie.ca,Roscoeview


In [7]:
# Posts DataFrame Creation
if posts_data:

    df_posts = pd.DataFrame(posts_data)

    df_posts = df_posts[
        ['userId', 'id', 'title', 'body']
    ]

    df_posts.columns = [
        'user_id',
        'post_id',
        'title',
        'body'
    ]

    print("Posts DataFrame:")
    display(df_posts.head())

Posts DataFrame:


,user_id,post_id,title,body
0,1,1,sunt aut facere repellat provident occaecati e...,quia et suscipit\nsuscipit recusandae consequu...
1,1,2,qui est esse,est rerum tempore vitae\nsequi sint nihil repr...
2,1,3,ea molestias quasi exercitationem repellat qui...,et iusto sed quo iure\nvoluptatem occaecati om...
3,1,4,eum et est occaecati,ullam et saepe reiciendis voluptatem adipisci\...
4,1,5,nesciunt quas odio,repudiandae veniam quaerat sunt sed\nalias aut...


In [8]:
# Create Messy Local CSV File

messy_data = {
    "user_id": [1, 2, 3, 4, 5, 5, None, 8],
    
    "name": [
        "  Leanne Graham ",
        "Ervin Howell",
        "CLEMENTINE BAUCH",
        "Patricia Lebsack",
        "Chelsey Dietrich",
        "Chelsey Dietrich",
        " ",
        "Nicholas Runolfsdottir"
    ],
    
    "email": [
        "leanne@gmail.com",
        "ervin@gmail.com",
        "clementine@gmail.com",
        "patricia@gmail.com",
        "chelsey@gmail.com",
        "chelsey@gmail.com",
        None,
        "nicholas@gmail.com "
    ],
    
    "city": [
        "Kathmandu",
        "Pokhara",
        "Biratnagar",
        " Lalitpur ",
        "Bhaktapur",
        "Bhaktapur",
        "Butwal",
        "Dharan"
    ],
    
    "age": [
        21,
        22,
        500,     # outlier
        19,
        None,
        19,
        20,
        "25 "
    ]
}

# Convert dictionary -> DataFrame
df_messy = pd.DataFrame(messy_data)

# Save CSV
df_messy.to_csv("messy_users.csv", index=False)

print("Messy CSV file created successfully!")

# Preview
display(df_messy)

Messy CSV file created successfully!


,user_id,name,email,city,age
0,1.0,Leanne Graham,leanne@gmail.com,Kathmandu,21
1,2.0,Ervin Howell,ervin@gmail.com,Pokhara,22
2,3.0,CLEMENTINE BAUCH,clementine@gmail.com,Biratnagar,500
3,4.0,Patricia Lebsack,patricia@gmail.com,Lalitpur,19
4,5.0,Chelsey Dietrich,chelsey@gmail.com,Bhaktapur,None
5,5.0,Chelsey Dietrich,chelsey@gmail.com,Bhaktapur,19
6,NaN,,NaN,Butwal,20
7,8.0,Nicholas Runolfsdottir,nicholas@gmail.com,Dharan,25


In [9]:
# Load Messy CSV

df_local = pd.read_csv("messy_users.csv")

print("Local Messy CSV Data:")
display(df_local.head(6))

Local Messy CSV Data:


,user_id,name,email,city,age
0,1.0,Leanne Graham,leanne@gmail.com,Kathmandu,21.0
1,2.0,Ervin Howell,ervin@gmail.com,Pokhara,22.0
2,3.0,CLEMENTINE BAUCH,clementine@gmail.com,Biratnagar,500.0
3,4.0,Patricia Lebsack,patricia@gmail.com,Lalitpur,19.0
4,5.0,Chelsey Dietrich,chelsey@gmail.com,Bhaktapur,NaN
5,5.0,Chelsey Dietrich,chelsey@gmail.com,Bhaktapur,19.0


In [10]:
#  Merge + Conflict Resolution

# Merge on email
merged_df = pd.merge(
    df_users,
    df_local,
    on="email",
    how="outer",
    suffixes=("_api", "_local")
)

print("Merged DataFrame:")
display(merged_df.head())


# Conflict Resolution Logic

"""
Decision:
If same email exists in both sources but names are different,
API data will win because API is considered more reliable
than local messy CSV data.
"""

# Keep API values first, otherwise use local values
merged_df['user_id'] = merged_df['user_id_api'].combine_first(
    merged_df['user_id_local']
)

merged_df['name'] = merged_df['name_api'].combine_first(
    merged_df['name_local']
)

merged_df['city'] = merged_df['city_api'].combine_first(
    merged_df['city_local']
)

# Select final cleaned columns
final_df = merged_df[
    ['user_id', 'name', 'email', 'city', 'age']
]

print("After Conflict Resolution:")
display(final_df.head())

Merged DataFrame:


,user_id_api,name_api,email,city_api,user_id_local,name_local,city_local,age
0,9.0,Glenna Reichert,Chaim_McDermott@dana.io,Bartholomebury,NaN,NaN,NaN,NaN
1,4.0,Patricia Lebsack,Julianne.OConner@kory.org,South Elvis,NaN,NaN,NaN,NaN
2,6.0,Mrs. Dennis Schulist,Karley_Dach@jasper.info,South Christy,NaN,NaN,NaN,NaN
3,5.0,Chelsey Dietrich,Lucio_Hettinger@annie.ca,Roscoeview,NaN,NaN,NaN,NaN
4,3.0,Clementine Bauch,Nathan@yesenia.net,McKenziehaven,NaN,NaN,NaN,NaN


After Conflict Resolution:


,user_id,name,email,city,age
0,9.0,Glenna Reichert,Chaim_McDermott@dana.io,Bartholomebury,NaN
1,4.0,Patricia Lebsack,Julianne.OConner@kory.org,South Elvis,NaN
2,6.0,Mrs. Dennis Schulist,Karley_Dach@jasper.info,South Christy,NaN
3,5.0,Chelsey Dietrich,Lucio_Hettinger@annie.ca,Roscoeview,NaN
4,3.0,Clementine Bauch,Nathan@yesenia.net,McKenziehaven,NaN


In [11]:
# DATA CLEANING

print("Before Cleaning:")
print(final_df.info())

# -------------------------------
# 1. Handle Null Values

# Remove rows where email is missing
final_df = final_df.dropna(subset=['email'])

# Fill missing city values
final_df['city'] = final_df['city'].fillna('Unknown')

# Fill missing age with median age
final_df['age'] = final_df['age'].fillna(final_df['age'].median())
print("\nNull values handled")

# -------------------------------
# 2. Remove Duplicates

# Remove duplicate emails
final_df = final_df.drop_duplicates(subset=['email'])

# -------------------------------
# 3. Fix Casing

# Proper title case for names and cities
final_df['name'] = final_df['name'].str.title()
final_df['city'] = final_df['city'].str.title()

# Lowercase emails
final_df['email'] = final_df['email'].str.lower()

print("Casing fixed")

# -------------------------------
# 4. Fix Data Types

# Convert user_id to integer
final_df['user_id'] = pd.to_numeric(
    final_df['user_id'],
    errors='coerce' # Any value that cannot be converted to integer becomes NaN because of 'coerce'
).astype('Int64')

# Convert age to numeric
final_df['age'] = pd.to_numeric(
    final_df['age'],
    errors='coerce'
)
print("Data types fixed")

# -------------------------------
# 5. Remove Extra Whitespace

final_df['name'] = final_df['name'].str.strip()
final_df['email'] = final_df['email'].str.strip()
final_df['city'] = final_df['city'].str.strip()

print("Whitespace cleaned")

# -------------------------------
# 6. Handle Outliers 

# Keep only realistic ages
final_df = final_df[
    (final_df['age'] >= 10) &
    (final_df['age'] <= 100)
]
print("Outliers removed")

# -------------------------------
# Final Result

print("\nAfter Cleaning:")
print(final_df.info())

display(final_df.head())

Before Cleaning:
<class 'pandas.DataFrame'>
RangeIndex: 18 entries, 0 to 17
Data columns (total 5 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   user_id  17 non-null     float64
 1   name     18 non-null     str    
 2   email    17 non-null     str    
 3   city     18 non-null     str    
 4   age      7 non-null      float64
dtypes: float64(2), str(3)
memory usage: 848.0 bytes
None

Null values handled
Casing fixed
Data types fixed
Whitespace cleaned
Outliers removed

After Cleaning:
<class 'pandas.DataFrame'>
Index: 15 entries, 0 to 16
Data columns (total 5 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   user_id  15 non-null     Int64  
 1   name     15 non-null     str    
 2   email    15 non-null     str    
 3   city     15 non-null     str    
 4   age      15 non-null     float64
dtypes: Int64(1), float64(1), str(3)
memory usage: 735.0 bytes
None


,user_id,name,email,city,age
0,9,Glenna Reichert,chaim_mcdermott@dana.io,Bartholomebury,21.5
1,4,Patricia Lebsack,julianne.oconner@kory.org,South Elvis,21.5
2,6,Mrs. Dennis Schulist,karley_dach@jasper.info,South Christy,21.5
3,5,Chelsey Dietrich,lucio_hettinger@annie.ca,Roscoeview,21.5
4,3,Clementine Bauch,nathan@yesenia.net,Mckenziehaven,21.5


In [12]:
# Save Final DataFrame to CSV
final_df.to_csv("final_unified_data.csv", index=False)

#Load Data into MySQL db
import mysql.connector

conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="root"
)

cursor = conn.cursor()

cursor.execute("CREATE DATABASE IF NOT EXISTS etl_db")
cursor.execute("USE etl_db")

cursor.execute("""
CREATE TABLE IF NOT EXISTS unified_users (
    user_id INT,
    name VARCHAR(255),
    email VARCHAR(255) UNIQUE,
    city VARCHAR(255),
    age INT
)
""")


# UPSERT QUERY (No duplicates ever)
query = """
INSERT INTO unified_users (user_id, name, email, city, age)
VALUES (%s, %s, %s, %s, %s)
ON DUPLICATE KEY UPDATE
    name = VALUES(name),
    city = VALUES(city),
    age = VALUES(age)
"""

for _, row in final_df.iterrows():
    cursor.execute(query, (
        int(row['user_id']) if pd.notnull(row['user_id']) else None,
        row['name'],
        row['email'],
        row['city'],
        int(row['age']) if pd.notnull(row['age']) else None
    ))
conn.commit()
print("Data loaded into MySQL successfully!")

cursor.close()
conn.close()
print("MySQL connection closed")


Data loaded into MySQL successfully!
MySQL connection closed
